# Google Shopping – 3 ofertas mais baratas por item

Lê uma lista de itens de uma **Google Sheets**, busca cada um no **Google Shopping** e grava as **3 ofertas mais baratas** (preço, loja, título e link) numa nova aba da mesma planilha.

**Como usar**
1. Preencha a célula **⚙️ Configuração** (link da planilha, aba e coluna).
2. Menu **Ambiente de execução → Executar tudo**.
3. Autorize o acesso à sua conta Google quando o Colab pedir.
4. O resultado aparece na aba **`Resultados`** da planilha.

> ⚠️ **Limitação:** é raspagem direta do Google. O Google pode mostrar captcha e bloquear depois de muitas buscas seguidas. Nesse caso o item fica com status `BLOQUEADO` – espere alguns minutos e rode de novo (só os itens pendentes são refeitos se `PULAR_JA_FEITOS = True`).

## ⚙️ Configuração

In [ ]:
# Link (ou ID) da planilha do Google
PLANILHA_URL = "https://docs.google.com/spreadsheets/d/COLE_O_ID_AQUI/edit"

ABA_ITENS      = "Itens"       # aba onde está a lista de itens
COLUNA_ITEM    = "Item"        # nome do cabeçalho da coluna com os nomes dos itens
ABA_RESULTADOS = "Resultados"  # aba de saída (é criada se não existir)

QTD_OFERTAS     = 3      # quantas ofertas mais baratas trazer
FILTRAR_TITULO  = True   # descarta anúncios cujo título não tem a maioria das palavras do item
PULAR_JA_FEITOS = True   # não refaz itens que já estão OK na aba de resultados
PAUSA_SEG       = (4, 9) # pausa aleatória entre buscas (min, max) – reduz risco de captcha

## 1. Instalação (só na primeira execução da sessão)

In [ ]:
!pip install -q playwright gspread
!playwright install --with-deps chromium > /dev/null 2>&1
print("OK")

## 2. Acesso à Google Sheets

In [ ]:
import gspread
from google.colab import auth
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

planilha = gc.open_by_url(PLANILHA_URL) if PLANILHA_URL.startswith("http") else gc.open_by_key(PLANILHA_URL)
linhas = planilha.worksheet(ABA_ITENS).get_all_records()
itens = [str(l.get(COLUNA_ITEM, "")).strip() for l in linhas]
itens = list(dict.fromkeys(i for i in itens if i))  # remove vazios e duplicados
print(f"{len(itens)} itens encontrados em '{ABA_ITENS}'")

## 3. Funções de busca

In [ ]:
import re, unicodedata
from urllib.parse import quote_plus, urlparse, parse_qs

USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36")

def parse_preco(texto):
    """'R$ 1.234,56' -> 1234.56 (None se não achar)."""
    m = re.search(r"R\$\s*(\d{1,3}(?:\.\d{3})*|\d+)(?:,(\d{2}))?", texto or "")
    if not m:
        return None
    return float(m.group(1).replace(".", "") + "." + (m.group(2) or "00"))

def _normaliza(s):
    s = unicodedata.normalize("NFKD", s.lower())
    return "".join(c for c in s if not unicodedata.combining(c))

def titulo_confere(item, titulo, minimo=0.6):
    """True se o título contém pelo menos `minimo` (60%) das palavras do item."""
    palavras = [p for p in re.findall(r"\w+", _normaliza(item)) if len(p) > 2]
    if not palavras:
        return True
    t = _normaliza(titulo)
    return sum(p in t for p in palavras) / len(palavras) >= minimo

def limpa_link(href):
    """Converte links de redirecionamento do Google (/url?q=...) no link da loja."""
    if not href:
        return ""
    if href.startswith("/"):
        href = "https://www.google.com" + href
    u = urlparse(href)
    if u.path == "/url" and ("q=" in u.query or "url=" in u.query):
        q = parse_qs(u.query)
        return (q.get("q") or q.get("url") or [href])[0]
    return href

# Roda dentro da página: acha cada elemento com preço "R$", sobe até o "card"
# do produto e extrai título, preço, loja e link. Não depende de classes CSS
# do Google (que mudam com frequência), só da estrutura texto + link.
EXTRAIR_JS = r"""
() => {
  const precoRe = /R\$\s?\d[\d.]*(,\d{2})?/;
  const ruido = /^(R\$|frete|entrega|\d+([.,]\d+)?\s*(\(|$)|avalia|compar|mais lojas|loja(s)? (oficial|verificada)|usado|\+|em até|à vista|ver |patrocinad|anúncio|oferta|desconto|-?\d+%)/i;
  const visto = new Set(), out = [];
  const folhas = [...document.querySelectorAll('span,div,b,strong')]
    .filter(e => e.children.length === 0 && precoRe.test(e.textContent) && e.textContent.length < 40);
  for (const f of folhas) {
    if (getComputedStyle(f).textDecorationLine.includes('line-through')) continue; // preço antigo riscado
    let card = f.parentElement;
    for (let i = 0; i < 10 && card; i++, card = card.parentElement) {
      const linhas = card.innerText.split('\n').map(s => s.trim()).filter(Boolean);
      if (linhas.length >= 3 && (card.matches('a[href]') || card.querySelector('a[href]') || card.closest('a[href]'))) break;
    }
    if (!card || visto.has(card)) continue;
    // ignora containers grandes (lista inteira) – um card tem só 1 ou 2 preços
    const precos = card.innerText.match(new RegExp(precoRe.source, 'g')) || [];
    if (precos.length > 3) continue;
    visto.add(card);
    const linhas = card.innerText.split('\n').map(s => s.trim()).filter(Boolean);
    const iPreco = linhas.findIndex(l => l.includes(f.textContent.trim()));
    const candidatos = linhas.filter(l => !precoRe.test(l) && !ruido.test(l));
    const titulo = candidatos.slice().sort((a, b) => b.length - a.length)[0] || '';
    let loja = '';
    for (const l of linhas.slice(iPreco + 1)) {
      if (!precoRe.test(l) && !ruido.test(l) && l !== titulo && l.length <= 40) { loja = l; break; }
    }
    const links = [...card.querySelectorAll('a[href]')].map(a => a.href);
    if (card.closest('a[href]')) links.unshift(card.closest('a[href]').href);
    const externo = links.find(h => !/google\./.test(new URL(h, location.href).hostname));
    out.push({titulo, preco_txt: f.textContent.trim(), loja, link: externo || links[0] || ''});
  }
  return out;
}
"""

async def buscar_item(page, item, qtd=3, filtrar=True):
    """Retorna (status, [ofertas mais baratas]) para um item."""
    url = ("https://www.google.com/search?tbm=shop&hl=pt-BR&gl=br&tbs=p_ord:p&q=" + quote_plus(item))
    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=45000)
        # aceita aviso de cookies, se aparecer
        for txt in ["Aceitar tudo", "Accept all"]:
            btn = page.get_by_role("button", name=txt)
            if await btn.count():
                await btn.first.click(); break
        await page.wait_for_timeout(2500)
        html = (await page.content()).lower()
        if "/sorry/" in page.url or "unusual traffic" in html or "tráfego incomum" in html or "recaptcha" in html:
            return "BLOQUEADO", []
        await page.mouse.wheel(0, 3000)  # carrega mais cards (lazy load)
        await page.wait_for_timeout(1500)
        brutos = await page.evaluate(EXTRAIR_JS)
    except Exception as e:
        return f"ERRO: {type(e).__name__}", []

    ofertas, vistos = [], set()
    for o in brutos:
        preco = parse_preco(o["preco_txt"])
        if preco is None or not o["titulo"]:
            continue
        if filtrar and not titulo_confere(item, o["titulo"]):
            continue
        chave = (o["titulo"].lower(), o["loja"].lower(), preco)
        if chave in vistos:
            continue
        vistos.add(chave)
        link = limpa_link(o["link"]) or ("https://www.google.com/search?tbm=shop&q=" + quote_plus(o["titulo"]))
        ofertas.append({"titulo": o["titulo"], "preco": preco, "loja": o["loja"], "link": link})

    ofertas.sort(key=lambda o: o["preco"])
    return ("OK" if ofertas else "SEM RESULTADO"), ofertas[:qtd]

## 4. Executar busca

In [ ]:
import asyncio, random
import pandas as pd
from playwright.async_api import async_playwright

COLUNAS = ["Item", "Posição", "Preço (R$)", "Loja", "Título do anúncio", "Link", "Status"]

# Carrega resultados anteriores (para pular itens já feitos)
try:
    ws_out = planilha.worksheet(ABA_RESULTADOS)
    anteriores = pd.DataFrame(ws_out.get_all_records())
except gspread.WorksheetNotFound:
    ws_out = planilha.add_worksheet(ABA_RESULTADOS, rows=1000, cols=len(COLUNAS))
    anteriores = pd.DataFrame(columns=COLUNAS)

feitos = set()
if PULAR_JA_FEITOS and not anteriores.empty and "Status" in anteriores:
    feitos = set(anteriores.loc[anteriores["Status"] == "OK", "Item"])
    anteriores = anteriores[anteriores["Item"].isin(feitos)]
else:
    anteriores = pd.DataFrame(columns=COLUNAS)

pendentes = [i for i in itens if i not in feitos]
print(f"{len(pendentes)} itens para buscar ({len(feitos)} já feitos)")

novas = []
async with async_playwright() as pw:
    browser = await pw.chromium.launch(headless=True, args=["--disable-blink-features=AutomationControlled"])
    ctx = await browser.new_context(locale="pt-BR", user_agent=USER_AGENT, viewport={"width": 1366, "height": 900})
    page = await ctx.new_page()
    bloqueado = False
    for n, item in enumerate(pendentes, 1):
        if bloqueado:
            novas.append({"Item": item, "Status": "BLOQUEADO"}); continue
        status, ofertas = await buscar_item(page, item, QTD_OFERTAS, FILTRAR_TITULO)
        print(f"[{n}/{len(pendentes)}] {item}: {status} ({len(ofertas)} ofertas)")
        if status == "BLOQUEADO":
            bloqueado = True  # para de insistir para não piorar o bloqueio
        if ofertas:
            for pos, o in enumerate(ofertas, 1):
                novas.append({"Item": item, "Posição": pos, "Preço (R$)": o["preco"], "Loja": o["loja"],
                              "Título do anúncio": o["titulo"], "Link": o["link"], "Status": status})
        else:
            novas.append({"Item": item, "Status": status})
        await asyncio.sleep(random.uniform(*PAUSA_SEG))
    await browser.close()

resultado = pd.concat([anteriores, pd.DataFrame(novas)], ignore_index=True).reindex(columns=COLUNAS)
for col in ["Posição", "Preço (R$)"]:
    resultado[col] = pd.to_numeric(resultado[col], errors="coerce")
ordem = {nome: i for i, nome in enumerate(itens)}
resultado["_ordem"] = resultado["Item"].map(ordem)
resultado = resultado.sort_values(["_ordem", "Posição"]).drop(columns="_ordem")
resultado["Posição"] = resultado["Posição"].astype("Int64")

valores = resultado.astype(object).where(resultado.notna(), "").values.tolist()
ws_out.clear()
ws_out.update([COLUNAS] + valores, value_input_option="RAW")
print(f"\nPronto! Resultado gravado na aba '{ABA_RESULTADOS}'.")
resultado

## Status possíveis
| Status | Significado |
|---|---|
| `OK` | Ofertas encontradas |
| `SEM RESULTADO` | Google não retornou ofertas (ou nenhuma passou no filtro de título) |
| `BLOQUEADO` | Google pediu captcha. Espere alguns minutos e rode a célula 4 de novo |
| `ERRO: ...` | Falha inesperada (página não carregou, etc.) |